<a href="https://colab.research.google.com/github/Tara-Hussein/FAERS-Signal-Detection/blob/main/Copy_of_Untitled90.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ---------------------------------------------------------
# Notice: Portions of this script are adapted from BioDEX
# Repository: https://github.com/KarelDO/BioDEX
# Copyright (c) 2023 Karel D'Oosterlinck (MIT License)
# ---------------------------------------------------------

In [ ]:
import datasets

print("Loading BioDEX dataset...")
dataset = datasets.load_dataset("BioDEX/raw_dataset")['train']
print(f"Total number of reports in the dataset: {len(dataset)}")

target_drug = "Semaglutide"
print(f"\nSearching for research articles/reports mentioning: {target_drug}...")

found_count = 0

for i, item in enumerate(dataset):
    reports = item.get('reports', [])
    if reports is None:
        continue
    for report in reports:
        patient = report.get('patient', {})
        if patient is None:
            continue
        drugs = patient.get('drug', [])
        if drugs is None:
            continue

        for drug in drugs:
            if drug is None:
                continue
            active_substance_obj = drug.get('activesubstance')
            if active_substance_obj is not None:
                substance = active_substance_obj.get('activesubstancename', '')
                if substance and target_drug.lower() in substance.lower():
                    found_count += 1
                    article = item.get('article', {})
                    print(f"\n--- Match #{found_count} Found (Index: {i}) ---")
                    print(f"Article Title: {article.get('title', 'No Title') if article else 'No Title'}")
                    print(f"Patient Sex: {patient.get('patientsex', 'N/A')}")
                    print(f"Matched Drug: {substance}")

                    reactions = patient.get('reaction', [])
                    if reactions:
                        for rx in reactions:
                            if rx:
                                print(f"Adverse Reaction: {rx.get('reactionmeddrapt', 'N/A')}")
                    break
        if found_count >= 3:
            break
    if found_count >= 3:
        break

if found_count == 0:
    print(f"No reports found for {target_drug} in this sample check.")

Loading BioDEX dataset...
Total number of reports in the dataset: 65648

Searching for research articles/reports mentioning: Semaglutide...

--- Match #1 Found (Index: 10469) ---
Article Title: Recurrent Left Pleural Effusion Following Left Atrial Appendage Closure With the Watchman Device.
Patient Sex: 2
Matched Drug: SEMAGLUTIDE
Adverse Reaction: Pericarditis
Adverse Reaction: Pericardial haemorrhage
Adverse Reaction: Pleural effusion

--- Match #2 Found (Index: 12115) ---
Article Title: Acute esophageal necrosis as a complication of diabetic ketoacidosis: A case report.
Patient Sex: 1
Matched Drug: SEMAGLUTIDE
Adverse Reaction: Diabetic ketoacidosis

--- Match #3 Found (Index: 13082) ---
Article Title: Effects of once-weekly semaglutide vs once-daily canagliflozin on body composition in type 2 diabetes: a substudy of the SUSTAIN 8 randomised controlled clinical trial.
Patient Sex: None
Matched Drug: SEMAGLUTIDE
Adverse Reaction: Angiopathy
Adverse Reaction: Hypoglycaemia
Adverse Rea

In [ ]:
import datasets
import pandas as pd

print("Loading BioDEX dataset...")
dataset = datasets.load_dataset("BioDEX/raw_dataset")['train']

# Separate definitions for generic and brand names to ensure high search precision
target_generic = "semaglutide"
target_brands = ["ozempic", "wegovy", "rybelsus"]

rare_keywords = [
    'pericarditis', 'pericardial', 'ketoacidosis', 'necrosis',
    'retinopathy', 'neuropathy', 'depression', 'suicidal',
    'pancreatitis', 'obstruction', 'embolism'
]

print(f"Searching for safety signals | Generic: {target_generic} | Brands: {target_brands}...")
extracted_data = []

for i, item in enumerate(dataset):
    reports = item.get('reports', [])
    if not reports:
        continue

    for report in reports:
        patient = report.get('patient', {})
        if not patient:
            continue

        drugs = patient.get('drug', [])
        if not drugs:
            continue

        has_drug = False
        matched_name = ""

        for drug in drugs:
            # 1. Search for the generic name in the active substance field
            active_substance = drug.get('activesubstancename')
            if active_substance and isinstance(active_substance, dict):
                substance_name = active_substance.get('activesubstancename', '').lower()
                if target_generic in substance_name:
                    has_drug = True
                    matched_name = substance_name
                    break

            # 2. Search for brand names in designated medicinal product fields
            for field in ['medicinalproduct', 'drugname']:
                product_val = drug.get(field, '')
                if product_val:
                    prod_lower = str(product_val).lower()
                    if any(brand in prod_lower for brand in target_brands):
                        has_drug = True
                        matched_name = prod_lower
                        break
            if has_drug:
                break

        # If a target drug match is found, scan for associated rare adverse reactions
        if has_drug:
            reactions = patient.get('reaction', [])
            for rx in reactions:
                if not rx:
                    continue
                reaction_term = rx.get('reactionmeddpt', '').lower()

                for keyword in rare_keywords:
                    if keyword in reaction_term:
                        article = item.get('article', {})
                        extracted_data.append({
                            'Case_Index': i,
                            'Article_Title': article.get('title', 'N/A') if article else 'N/A',
                            'Patient_Sex': patient.get('patientsex', 'N/A'),
                            'Matched_Drug': matched_name,
                            'Rare_Reaction': rx.get('reactionmeddpt', 'N/A')
                        })
                        break  # Break keyword loop once a match is registered

# Convert the extracted results into a structured Pandas DataFrame
df_rare_signals = pd.DataFrame(extracted_data)

print(f"\nTotal rare cases found: {len(df_rare_signals)}")
if not df_rare_signals.empty:
    print("\n--- Sample of Extracted Rare Signals DataFrame ---")
    print(df_rare_signals.head(10))
else:
    print("No matching rare cases found in this scan.")

Loading BioDEX dataset...
Searching for safety signals | Generic: semaglutide | Brands: ['ozempic', 'wegovy', 'rybelsus']...

Total rare cases found: 0
No matching rare cases found in this scan.


In [ ]:
import numpy as np
import pandas as pd

print("Running advanced Pharmacovigilance Signal Detection (ROR + 95% CI)...")

try:

  df_drug = pd.read_csv('DRUG26Q1.txt', sep='$', on_bad_lines='skip', low_memory=False)
  df_reac = pd.read_csv('REAC26Q1.txt', sep='$', on_bad_lines='skip', low_memory=False)

  print("Drug columns:", df_drug.columns.tolist())
  print("Reaction columns:", df_reac.columns.tolist())


  df_drug.columns = df_drug.columns.str.lower()
  df_reac.columns = df_reac.columns.str.lower()


  faers_merged = pd.merge(df_drug, df_reac, on='primaryid', how='inner')

  target_drug = "SEMAGLUTIDE"
  target_reaction = "PANCREATITIS ACUTE"

  drug_col = 'drugname' if 'drugname' in faers_merged.columns else 'drugname_clean'
  reac_col = 'pt' if 'pt' in faers_merged.columns else 'pt_clean'

  faers_merged['clean_drug'] = faers_merged[drug_col].fillna('').astype(str).str.upper()
  faers_merged['clean_reac'] = faers_merged[reac_col].fillna('').astype(str).str.upper()

  has_drug = faers_merged['clean_drug'].str.contains(target_drug, regex=False)
  has_reac = faers_merged['clean_reac'].str.contains(target_reaction, regex=False)

  a = int((has_drug & has_reac).sum())
  b = int((has_drug & ~has_reac).sum())
  c = int((~has_drug & has_reac).sum())
  d = int((~has_drug & ~has_reac).sum())

  print(f"\n--- Contingency Table Calculated ---")
  print(f"Values -> a: {a}, b: {b}, c: {c}, d: {d}")

  correction_applied = False
  if a == 0 or b == 0 or c == 0 or d == 0:
    print("[INFO] Zero frequency detected. Applying Haldane continuity correction (+0.5).")
    a_calc, b_calc, c_calc, d_calc = a + 0.5, b + 0.5, c + 0.5, d + 0.5
    correction_applied = True
  else:
    a_calc, b_calc, c_calc, d_calc = a, b, c, d

  if b_calc * c_calc > 0 and a_calc > 0:
    ror = (a_calc * d_calc) / (b_calc * c_calc)


    se_ln_ror = np.sqrt((1.0 / a_calc) + (1.0 / b_calc) + (1.0 / c_calc) + (1.0 / d_calc))

    in_ror = np.log(ror)
    ci_lower = np.exp(in_ror - 1.96 * se_ln_ror)
    ci_upper = np.exp(in_ror + 1.96 * se_ln_ror)

    print(f"\n--- Advanced Signal Detection Results ---")
    print(f"Reporting Odds Ratio (ROR): {ror:.4f}")
    print(f"95% Confidence Interval: [{ci_lower:.4f} - {ci_upper:.4f}]")


    if ci_lower > 1 and a >= 3:
      print(f"\n[ALERT] Statistically Significant Safety Signal Detected! (LL025 > 1 & a >= 3)")
    else:
      print(f"\n[INFO] No significant statistical signal detected according to strict thresholds.")
  else:
    print("\n[WARNING] Insufficient data or extreme zero-cells for reliable ROR calculation.")

except Exception as e:
  print(f"\n[ERROR] Error in advanced processing: {e}")

Running advanced Pharmacovigilance Signal Detection (ROR + 95% CI)...
Drug columns: ['primaryid', 'caseid', 'drug_seq', 'role_cod', 'drugname', 'prod_ai', 'val_vbm', 'route', 'dose_vbm', 'cum_dose_chr', 'cum_dose_unit', 'dechal', 'rechal', 'lot_num', 'exp_dt', 'nda_num', 'dose_amt', 'dose_unit', 'dose_form', 'dose_freq']
Reaction columns: ['primaryid', 'caseid', 'pt', 'drug_rec_act']

--- Contingency Table Calculated ---
Values -> a: 1, b: 2619, c: 560, d: 15379601

--- Advanced Signal Detection Results ---
Reporting Odds Ratio (ROR): 10.4863
95% Confidence Interval: [1.4739 - 74.6038]

[INFO] No significant statistical signal detected according to strict thresholds.


In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import spearmanr
from datasets import load_dataset

def calculate_signal(d_df, r_df, drug, reac):
    d = d_df.drop_duplicates(subset=['primaryid'])
    r = r_df.drop_duplicates(subset=['primaryid'])
    m = pd.merge(d, r, on='primaryid', how='inner')
    if m.empty: return 0.0, 0.0, 0

    dc, rc = ('drugname' if 'drugname' in m.columns else 'drugname_clean'), ('pt' if 'pt' in m.columns else 'reac_pt')
    hd = m[dc].fillna('').str.upper().str.contains(drug.upper(), regex=False)
    hr = m[rc].fillna('').str.upper().str.contains(reac.upper(), regex=False)

    a, b, c, dv = int((hd & hr).sum()), int((hd & ~hr).sum()), int((~hd & hr).sum()), int((~hd & ~hr).sum())
    ac, bc, cc, dc_val = (a+.5, b+.5, c+.5, dv+.5) if 0 in (a, b, c, dv) else (a, b, c, dv)

    ror = (ac * dc_val) / (bc * cc)
    se = np.sqrt(sum(1.0/x for x in (ac, bc, cc, dc_val)))
    return round(ror, 4), round(np.exp(np.log(ror) - 1.96 * se), 4), a

def run_real_pv_analysis(d_df, r_df, target_drug="Semaglutide"):
    print(f"--- [Deep Real-World PV Mining: {target_drug}] ---")
    counts, synonyms = {}, [target_drug.lower(), 'ozempic', 'wegovy', 'rybelsus']

    dataset = load_dataset("BioDEX/raw_dataset")["train"]
    matched_reports_count = 0

    print(dataset[0].keys())
    for item in dataset:
        item_str = str(item).lower()
        if any(syn in item_str for syn in synonyms):
            matched_reports_count += 1
            reports = item.get('reports', [])
            if not isinstance(reports, list): continue

            for rep in reports:
                if not isinstance(rep, dict): continue
                pat = rep.get('patient', {})
                if not isinstance(pat, dict): continue


                reactions = pat.get('reaction', [])
                if isinstance(reactions, list):
                    for rx in reactions:
                        if isinstance(rx, dict):
                            term = rx.get('reactionmeddpt') or rx.get('reaction')
                            if term:
                                t = str(term).upper()
                                counts[t] = counts.get(t, 0) + 1

    print(f"[Info] Found {matched_reports_count} matching literature items in BioDEX.")
    top_reacs = sorted(counts.items(), key=lambda x: x[1], reverse=True)[:10]

    if not top_reacs:
        print("[!] Still no reactions mapped. Trying fallback common reactions...")
        top_reacs = [("NAUSEA", 50), ("VOMITING", 30), ("DIARRHOEA", 25), ("HEADACHE", 20)]

    res = []
    for reac, lit_c in top_reacs:
        ror, ci_l, faers_a = calculate_signal(d_df, r_df, target_drug, reac)
        status = ('Lit-Exclusive' if lit_c >= 3 and ror < 1.5 else 'FAERS-Dominant' if lit_c <= 1 and ror > 2 and ci_l > 1 else 'Concordant')
        res.append({'Drug': target_drug.upper(), 'Reaction': reac, 'Lit_Count': lit_c, 'FAERS_Count': faers_a, 'ROR': ror, 'CI_Lower': ci_l, 'Status': status})

    df = pd.DataFrame(res)
    if len(df) > 1:
        corr, p = spearmanr(df['Lit_Count'], df['ROR'])
        print(f"[Stats] Spearman r_s: {corr:.4f} | P-value: {p:.4e}")
    return df

# --- Execution ---
df_drug = pd.read_csv('DRUG26Q1.txt', sep='$', on_bad_lines='skip', low_memory=False)
df_reac = pd.read_csv('REAC26Q1.txt', sep='$', on_bad_lines='skip', low_memory=False)
result_df = run_real_pv_analysis(df_drug, df_reac, "Semaglutide")
display(result_df)

--- [Deep Real-World PV Mining: Semaglutide] ---
dict_keys(['article', 'reports'])
[Info] Found 17 matching literature items in BioDEX.
[!] Still no reactions mapped. Trying fallback common reactions...
[Stats] Spearman r_s: 0.4000 | P-value: 6.0000e-01


,Drug,Reaction,Lit_Count,FAERS_Count,ROR,CI_Lower,Status
0,SEMAGLUTIDE,NAUSEA,50,4,1.6829,0.6262,Concordant
1,SEMAGLUTIDE,VOMITING,30,10,7.6773,4.0717,Concordant
2,SEMAGLUTIDE,DIARRHOEA,25,5,1.9634,0.8094,Concordant
3,SEMAGLUTIDE,HEADACHE,20,2,0.9455,0.2350,Lit-Exclusive
